# Projeto Aplicado — Sprint 1 · Base LLVE (R05)
**Segmentação de Clientes por Valor (RFM e Clusterização) para Otimização de Marketing e Retenção em E-commerce de Moda**

| R05 | Extrair, pseudonimizar e limpar a base da LLVE |
|---|---|
| Critério de aceite | Base LLVE tratada, com os compradores válidos e **sem dados pessoais** |

> **LGPD:** este notebook lê o arquivo bruto da pasta `DADOS LLVE` (privada, fora do GitHub) e grava apenas a versão pseudonimizada em `PROCESSADOS`. Nenhuma célula exibe nome, e-mail, telefone, documento ou endereço. **Não salve este notebook no GitHub.**

### 5.1 Ambiente e parâmetros

In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import hashlib, numpy as np, pandas as pd

BASE_DIR    = Path('/content/drive/MyDrive/PROJETO APLICADO CD/PROJETO APLICADO')
LLVE_DIR    = BASE_DIR / 'DADOS LLVE'                 # dados brutos — acesso restrito
ARQ_LLVE    = LLVE_DIR / 'clientes_llve_pa.csv'
OUTPUT_DIR  = BASE_DIR / 'PROCESSADOS'
FIG_DIR     = BASE_DIR / 'FIGURAS'
SEED = 42
PERCENTIL_CORTE = 0.999                              # mesma regra D4 da Olist, calculada na base da LLVE

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Arquivo encontrado:', ARQ_LLVE.exists())

Mounted at /content/drive
Arquivo encontrado: True


### 5.2 Chave secreta da pseudonimização
O identificador do cliente será um **hash SHA-256 do e-mail combinado com uma chave secreta**. Sem a chave, não é possível voltar ao e-mail; com ela, só você consegue reverter o código na Sprint 3 para montar os públicos.

**Como criar a chave (uma vez só):** no Colab, clique no ícone de chave 🔑 na barra lateral (*Secrets*) → *Adicionar novo secret* → nome `LLVE_SALT`, valor = uma frase longa e aleatória que só você conhece → ative *Acesso ao notebook*. A chave não fica gravada no notebook.

In [2]:
SALT = userdata.get('LLVE_SALT')
assert SALT and len(SALT) >= 16, 'Crie o secret LLVE_SALT com pelo menos 16 caracteres'

def pseudonimizar(valor):
    """Hash SHA-256 (chave + e-mail normalizado), truncado em 16 caracteres."""
    if pd.isna(valor) or str(valor).strip() == '':
        return np.nan
    texto = SALT + str(valor).strip().lower()
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()[:16]

print('Chave carregada · teste de hash:', pseudonimizar('exemplo@teste.com'))

Chave carregada · teste de hash: c4302846f715d47e


### 5.3 Leitura do arquivo e inspeção da estrutura (sem exibir dados)
A célula abaixo mostra **apenas** nomes de colunas, tipos, % de nulos e nº de valores distintos.

In [3]:
def ler_csv(caminho):
    for enc in ('utf-8', 'utf-8-sig', 'latin-1'):
        try:
            return pd.read_csv(caminho, sep=None, engine='python', encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError('Não foi possível ler o arquivo')

bruto, encoding = ler_csv(ARQ_LLVE)
print(f'{len(bruto):,} linhas × {bruto.shape[1]} colunas · encoding {encoding}')

estrutura = pd.DataFrame({
    'coluna': bruto.columns,
    'tipo': bruto.dtypes.astype(str).values,
    '% nulos': (bruto.isna().mean() * 100).round(1).values,
    'valores distintos': bruto.nunique().values,
})
estrutura

1,879 linhas × 24 colunas · encoding latin-1


,coluna,tipo,% nulos,valores distintos
0,Nome completo,object,0.10,1823
1,CPF/CNPJ,float64,20.60,1442
2,E-mail,object,0.00,1879
3,Telefone de Contato,float64,18.30,1481
4,Gênero,float64,100.00,0
5,Data de nascimento,float64,100.00,0
6,Endereço,object,45.30,843
7,Número,object,45.30,635
8,Complemento,object,54.60,721
9,Cidade,object,45.30,241
